<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m6_eval_suite.ipynb)
[![Course](https://img.shields.io/badge/Full%20Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-6)

</div>

# Module 6 — Eval Suite Design

**Level:** Advanced | **Time:** ~2h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-6)

### What you'll build
An eval harness with parallel trial execution, 3-grader composition (code, model, human), and pass@k / pass^k scoring.

### Key concepts
- **pass@k** = `1 − (1−p)^k` — probability at least 1 success in k trials (use for: one solution suffices)
- **pass^k** = `p^k` — probability all k trials succeed (use for: reliability-critical systems)
- **3 grader types**: code (fast, deterministic, brittle) → model (flexible, non-deterministic) → human (gold standard, slow)
- **LLM-as-judge calibration**: measure Cohen's κ against human labels; known failure modes: length bias, position bias, sycophancy
- **Saturation signal**: when pass@k → 100%, time to harden the eval set
- **Transcript inspection**: sample 10% of traces manually per run, look for grader misalignment

### Public benchmarks
SWE-Bench Verified (coding), GAIA (general), WebArena (browsing), τ-bench (tool use), AgentBench (multi-task)

---

In [ ]:
!pip install openai --quiet

In [ ]:
import os, json, math, time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Callable
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Metrics ───────────────────────────────────────────────────────────────────

def pass_at_k(p: float, k: int) -> float:
    """P(at least 1 success in k trials) = 1 - (1-p)^k"""
    return 1.0 - (1.0 - p) ** k

def pass_all_k(p: float, k: int) -> float:
    """P(all k trials succeed) = p^k"""
    return p ** k

print("pass@k demo:")
for p, k in [(0.7, 3), (0.5, 5), (0.9, 2)]:
    print(f"  p={p}, k={k}: pass@k={pass_at_k(p,k):.3f}, pass^k={pass_all_k(p,k):.3f}")

# ── Grader types ──────────────────────────────────────────────────────────────

@dataclass
class EvalTask:
    task_id: str
    prompt: str
    reference: str
    grader: str  # "code" | "model" | "hybrid"

def code_grader(prediction: str, reference: str) -> float:
    """Code-based grader: check key entities present. Fast, deterministic, brittle."""
    pred_lower = prediction.lower()
    ref_entities = [w for w in reference.lower().split() if len(w) > 4]
    if not ref_entities:
        return 0.0
    hits = sum(1 for e in ref_entities if e in pred_lower)
    return hits / len(ref_entities)

def model_grader(prediction: str, reference: str, task_prompt: str) -> float:
    """LLM-as-judge grader: flexible, handles nuance, non-deterministic."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": (
                "You are an eval judge. Score the prediction on faithfulness (does it match the reference?) "
                "and coverage (does it include all key facts?). "
                "Return JSON: {faithfulness: 0-1, coverage: 0-1, explanation: str}"
            )},
            {"role": "user", "content": f"Task: {task_prompt}\nReference: {reference}\nPrediction: {prediction}"}
        ],
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    return (result["faithfulness"] + result["coverage"]) / 2.0

# ── Eval runner ───────────────────────────────────────────────────────────────

class EvalSuite:
    def __init__(self, tasks: list[EvalTask], agent_fn: Callable[[str], str]):
        self.tasks = tasks
        self.agent_fn = agent_fn
        self.results: list[dict] = []

    def run_task(self, task: EvalTask, trial: int) -> dict:
        start = time.time()
        prediction = self.agent_fn(task.prompt)
        elapsed = time.time() - start

        if task.grader == "code":
            score = code_grader(prediction, task.reference)
        elif task.grader == "model":
            score = model_grader(prediction, task.reference, task.prompt)
        else:  # hybrid
            code_score = code_grader(prediction, task.reference)
            model_score = model_grader(prediction, task.reference, task.prompt)
            score = 0.4 * code_score + 0.6 * model_score

        return {
            "task_id": task.task_id, "trial": trial,
            "score": score, "pass": score >= 0.7,
            "elapsed": elapsed, "prediction": prediction[:200]
        }

    def run(self, k: int = 3) -> dict:
        all_results = []
        with ThreadPoolExecutor(max_workers=4) as ex:
            futures = [
                ex.submit(self.run_task, task, trial)
                for task in self.tasks
                for trial in range(k)
            ]
            all_results = [f.result() for f in futures]

        self.results = all_results

        # Aggregate pass@k and pass^k per task
        summary = {}
        for task in self.tasks:
            task_results = [r for r in all_results if r["task_id"] == task.task_id]
            passes = [r["pass"] for r in task_results]
            p_hat = sum(passes) / len(passes) if passes else 0.0
            summary[task.task_id] = {
                "p_hat": p_hat,
                "pass_at_k": pass_at_k(p_hat, k),
                "pass_all_k": pass_all_k(p_hat, k),
                "trials": passes
            }
        return summary


# ── Demo ──────────────────────────────────────────────────────────────────────

TASKS = [
    EvalTask("t1", "What is the capital of France?", "Paris", "code"),
    EvalTask("t2", "Explain what a transformer attention head does in 2 sentences.", "attention head queries keys values softmax weighted sum", "model"),
    EvalTask("t3", "What does RAG stand for?", "Retrieval-Augmented Generation", "code"),
]

def mock_agent(prompt: str) -> str:
    """Mock agent — replace with your real agent."""
    if "capital" in prompt.lower():
        return "The capital of France is Paris."
    if "attention" in prompt.lower():
        return "An attention head computes scaled dot-product attention over queries, keys, and values. It produces a weighted sum of values where weights come from softmax(QK^T/sqrt(d_k))."
    if "rag" in prompt.lower():
        import random
        return random.choice(["Retrieval-Augmented Generation", "Real-time Agentic Generation"])  # simulate noise
    return "I don't know."

suite = EvalSuite(TASKS, mock_agent)
summary = suite.run(k=3)

print("Eval results:")
for task_id, stats in summary.items():
    print(f"  {task_id}: p_hat={stats['p_hat']:.2f}, pass@3={stats['pass_at_k']:.3f}, pass^3={stats['pass_all_k']:.3f}, trials={stats['trials']}")

## Exercise

Write 5 eval tasks and implement Cohen's κ for grader calibration.

> **Interview question:** *"Your agent passes 95% of your eval suite. How do you know if the evals are measuring the right things?"*

In [ ]:
# Exercise: Write 5 eval tasks for the document research agent from Module 2
#
# For each task:
# 1. Write the task prompt (realistic research question)
# 2. Write a reference solution (verifiable answer with key entities)
# 3. Choose grader type (code | model | hybrid) and justify
# 4. Define partial credit criteria
# 5. Calculate pass@3 and pass^3 for an agent with p=0.8

p = 0.8
k = 3
print(f"For p={p}, k={k}:")
print(f"  pass@{k} = {pass_at_k(p, k):.3f}")
print(f"  pass^{k} = {pass_all_k(p, k):.3f}")
print(f"  Interpretation: agent almost certainly solves it ({pass_at_k(p,k)*100:.0f}%) but only {pass_all_k(p,k)*100:.0f}% of the time does it solve all {k} trials")

MY_TASKS = [
    # EvalTask("t1", "...", "...", "code"),
    # EvalTask("t2", "...", "...", "model"),
    # ... 5 tasks total
]

# Bonus: implement Cohen's kappa to measure model_grader vs human agreement
def cohens_kappa(grader_scores: list[float], human_scores: list[float], threshold: float = 0.7) -> float:
    """Measure inter-rater agreement between LLM grader and human labels."""
    # TODO: implement
    raise NotImplementedError